![image_1773389878407.png](./screenshots/image_1773389878407.png "image_1773389878407.png")

![image_1773389914334.png](./screenshots/image_1773389914334.png "image_1773389914334.png")

![image_1773390013116.png](./screenshots/image_1773390013116.png "image_1773390013116.png")

![image_1773390071332.png](./screenshots/image_1773390071332.png "image_1773390071332.png")

![image_1773390258798.png](./screenshots/image_1773390258798.png "image_1773390258798.png")

![image_1773390513906.png](./screenshots/image_1773390513906.png "image_1773390513906.png")

In [0]:
%pip install -r requirements.txt

In [0]:
import pypdf
import re

def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

reader = pypdf.PdfReader("Profile_Vinay_Tyagi_24.pdf")

resume_text = ""

for page in reader.pages:
  resume_text += page.extract_text()


clean_resume_text = clean_text(resume_text)
print(clean_resume_text)

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness, Guidelines
from mlflow.genai import scorer
from mlflow.entities import Feedback

eval_dataset = [
  {
    'inputs' : {'clean_resume_text': clean_resume_text},
    "expectations" : {"expected_response": '{"skills": ["python","ml","sklearn","linear_regression","machine_learning"]}'}
  }
]

def predict_fn(clean_resume_text) -> dict:
  # response = "python,ml,sklearn,linear regression,machine learning"
  import json
  return json.dumps({'skills': ["python","ml","sklearn","linear_regression","machine_learning"]})

  return response

@scorer
def minimum_five_skills(inputs,outputs, expectations):
  # response = len(outputs['response'].split(",")) >= 5
  import json
  try:
    skills_json = json.loads(outputs)
    if skills_json.get('skills'):
      if len(skills_json.get('skills')) > 5:
        return True
      else:
        return False
  except Exception as e:
    print(e)
  return False

@scorer
def is_json(outputs):
  import json
  try:
    skills_json = json.loads(outputs)
    return True
  except Exception as e:
    print(e)
  return False

    

scorers = [
  Correctness(),
  Guidelines(name = "coverage", guidelines = 'Are all the skills captured?'),
  minimum_five_skills,
  is_json
]  


mlflow.genai.evaluate(
  data = eval_dataset,
  predict_fn=predict_fn,
  scorers=scorers
)

In [0]:
with mlflow.start_run() as run:
    prompt_name = "mlflow_experiments.genai.resume_prompt"
    prompt = mlflow.genai.register_prompt(
        name=prompt_name,
        template="""You are a helpful assistant helping the user extract skills from the resume text. Here is the resume text: {{clean_resume_text}}.  Extract all the skills. The output should be this format {
            'skills': [] # list of skills
        }""",
        commit_message="Initial resume skills prompt"
    )
    print(f"Created version {prompt.version}")  # "Created version 1"

    # Set a production alias
    mlflow.genai.set_prompt_alias(
        name=prompt_name,
        alias="production",
        version=3
    )


In [0]:
versions = [1,3]
prompt_name = "mlflow_experiments.genai.resume_prompt"

for version in versions:
    prompt = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{prompt_name}/{version}")
    formatted_p = prompt.format(clean_resume_text=clean_resume_text)
    eval_dataset = [
        {
            'inputs' : {'clean_resume_text': formatted_p},
            "expectations" : {"expected_response": "python,ml,sklearn,linear regression,machine learning"}
        }
        ]
    # print(formatted_p[:100], "... ... ..."  ,formatted_p[-100:])
    response = predict_fn(formatted_p)
    mlflow.genai.evaluate(
        data = eval_dataset,
        predict_fn=predict_fn,
        scorers=scorers
        )
    # print("--"*100)